## ACNS 2026 Tutorial: Uncertainty analysis using sampling

In this notebook we will look at forward propagation from measurement uncertainty $x$ to uncertainty in $y = f(x)$.

The method is simple:

* generate a random sample for $x$
* evaluate $y = f(x)$ for each $x$
* calculate statistics on the collection $y$

Take away:

* the result you see is an accident of the measurement
* redo the measurement and you will get a different result
* report the range of values that are consistent with the measurement

Guide to Uncertainty in Measurement (GUM) Supplement 1: *Propagation of distributions using a Monte Carlo method* ([PDF from BIPM](https://www.bipm.org/documents/20126/2071204/JCGM_101_2008_E.pdf/325dcaad-c15a-407c-1105-8b7f322d651c))

[NIST TN 1297](https://www.nist.gov/pml/nist-technical-note-1297) *Guidelines for Evaluating and Expressing the Uncertainty of NIST Measurement Results* §4.3

In [ ]:
# Uncomment the following line to install packages used within this notebook
# %pip install numpy scipy corner uncertainties bumps matplotlib plotly

In [ ]:
# Prepare the environment
import numpy as np
import scipy.optimize
import scipy.stats  # for distributions
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import bumps.names as bp

# U(μ,σ) is handy for formatting numbers with uncertainties. For a compact
# format use f"{U(μ, σ):uS}", or f"{U(μ, σ):.2uS}" to force two digits.
# Example: u = U(3.141592, 0.003793) produces 3.142(4) or 3.1416(38)
from uncertainties import ufloat as U
import corner  # For corner plots showing correlations

# Random number generator to use for numpy
RNG = np.random.default_rng()

# You can change the plotly style from seaborn to something else
import plotly.io as pio

pio.templates.default = "simple_white"

%matplotlib inline

## Generating the sample

Use a random number generator (RNG) to generate the random sample

These are available in all scientific computing environments

Python has `numpy.random`, `scipy.stats`, `torch.distributions`, and others

Histograms are a primary tool for looking at a distribution

Rerun with different sample sizes. Every run is different

In [ ]:
# Forward uncertainty propagation starts by sampling from a distribution

# Exercise: Vary the location, scale and count to see how the histogram varies.

dist = scipy.stats.norm(0, 1)
# dist=scipy.stats.chi2(3)
# dist=scipy.stats.poisson(5)
# dist=scipy.stats.cauchy(loc=40, scale=0.35)

samples = dist.rvs(size=200)
fig = px.histogram(
    samples,
    title=f"Histogram of {dist.dist.name}{dist.args} (N={len(samples)})",
    marginal="rug",
    histnorm="probability density",
)
fig.update_layout(showlegend=False)
# use the quantile function dist.ppf to set a fixed plot range
x_limit = 0.001
fig.update_xaxes(range=[dist.ppf(x_limit), dist.ppf(1 - x_limit)])
fig.show()

### Monte Carlo Integration

Convert the sample into statistics using Monte Carlo integration

* **importance sampling**

Given a sample $X \sim g$ from probability distribution $g$, we can compute integrals over $f(x)g(x)$ using simple summation:

> $I = \int_\mathbb{R} f(x) g(x)\,\mathrm{d}x \approx \tfrac{1}{N} \sum_k f(x_k)$

This works because the density of points around $x_k$ is proportional to $g(x_k)$ so the sum will see more contributions from $f$ in high density regions.

* **mean**

The mean uses $f(x) = x$

> $μ = E[X] = \int x g(x)\,\mathrm{d}x \approx \tfrac{1}{N} \sum_k x_k$.

That looks right!

* **variance**

> $σ^2 = V[X] = E[(X - μ)²] = \int (x - μ)²g(x)\,\mathrm{d}x \approx \tfrac{1}{N} \sum_k (x_k - μ)^2$

This is correct if we use the known mean, but if we use the estimate $\hat μ$ it isn't quite right. It should be $\tfrac{1}{N-1}$ instead of $\tfrac{1}{N}$.

The problem is that the estimate $\hat μ = \tfrac{1}{N}\sum_j x_j$ uses the same $x$ value as the variance when $j = k$, so the results are correlated. Through careful bookkeeping (see [wikipedia](https://en.wikipedia.org/wiki/Variance#Biased_sample_variance)) you can create an unbiased expression, but for our purposes $N$ will be large enough the $\tfrac{N}{N-1}$ correction is insignificant.

* **cumulative density function (CDF)**

Next look at the cumulative density function. Using $f(x) = 1$ we get

> $G(a) = \int_{-\infty}^a g(x)\,\mathrm{d}x \approx \tfrac{1}{N} \sum_{x_k<a} 1$.

Our approximation for the cdf at $a$ is just the count of number of points below $a$ divided by $N$. You can examine the quality of the approximation by varying $N$ in the next cell.

* **quantile function**

The inverse of the CDF is the quantile function (also known as the percent point function, ppf, in scipy.stats). Given probability $p$, it returns $Q(p) = a$ such that $G(a) = p$.

This is easy to approximate with our sample: sort that values so that $x_i \le x_j$ for all $i < j$ and pick the sample at index $k = \lceil p N \rceil$

> $Q(p) = x_{\lceil p N \rceil}$.

This selected $x_k$ will be bigger than $p = k/N$ other points.

You can improve the approximation by interpolating between neighboring $x_k$ values when $p N$ is not an integer.

* **coverage intervals HCI and ETI**

The primary quantity we need for reporting uncertainty is the credible interval.†

For example, what range of $x$ values will cover 95% of the probability? This is equivalent to

> Interval(p) = $[a, b]$ such that $\int_a^b g(x)\,\mathrm{d}x \approx \tfrac{1}{N} \sum_{a\le x_k \le b} 1 = p$.

Determining the interval is easy with the quantile function. Since it ranges over probabilities, you just need to look at the difference between $Q(a)$ values at distance $p$ apart.

The primary choices are the equal-tailed interval (ETI) with

> ETI(p) = $[Q(a), Q(a+p)]$ for $a = (1-p)/2$

and the highest density interval (HDI) with

> HDI(p) = $[Q(a), Q(a+p)]$ for $a = \argmin_w Q(w+p) - Q(w)$.

* **median**

The median is simply

> median = $Q(0.5)$.

* **maximum likelihood estimate (MLE)**

Mode, or the maximum likelihood estimate (MLE), is more difficult. If you know $g(x)$ for your sample points then use

> $\text{MLE} = \argmax_x g(x)$

This works equally well if your distribution $g(x)$ is not normalized, which is often the case.

Traditional χ² fits return the MLE as the best fit value.

* **histogram**

If you just have the samples and you don't know $g$ then you can estimate the mode from the histogram. Each bin value is the integral

> $I_k = \int_{b_k}^{b_{k+1}} g(x)\,\mathrm{d}x$ for bin edges $b_1, b_2, ...$

The scikit learn package has kernel density estimators (KDE), which give a smoother density profile than the histogram.

* **entropy**

If you have $g(x)$ you can also compute the entropy

> $H(x) = \int g(x) \ln g(x)\,\mathrm{d}x \approx \tfrac{1}{N}\sum \ln g(x)$.

† GUM S1 uses "coverage interval" for "credible interval"; "symmetric intervale" for "equal-tailed interval" and "shortest coverage interval" for "highest density interval". Bayesian "credible intervals" are distinct from the "confidence intervals" of classical statistics.


In [ ]:
# Exercise: vary the sample size to see the effect on quantiles and the cdf


def do(dist, N=100):
    samples = np.sort(dist.rvs(size=N))  # dist.rvs is the scipy.stats random sampler
    q = np.linspace(0, 1, len(samples) + 2)[1:-1]
    x = np.linspace(samples[0], samples[-1], 400)
    cdf = dist.cdf(x)

    fig = px.line(
        x=samples, y=q, labels={"x": "x", "y": "p(x)"}, title=f"Empirical CDF ({N=}) for {dist.dist.name}{dist.args}"
    )
    fig.add_scatter(x=x, y=cdf, name="norm(0,1)")
    fig.update_layout(showlegend=False)
    fig.show()

    p = np.linspace(0, 1, 402)[1:-1]
    ppf = dist.ppf(p)

    fig = px.line(
        x=q,
        y=samples,
        labels={"x": "p(x)", "y": "x"},
        title=f"Empirical Quantiles ({N=}) for {dist.dist.name}{dist.args}",
    )
    fig.add_scatter(x=p, y=ppf, name="norm(0,1)")
    fig.update_layout(showlegend=False)
    fig.show()


do(dist=scipy.stats.norm(0, 1), N=200)
# do(dist=scipy.stats.chi2(3), N=200)
# do(dist=scipy.stats.poisson(45), N=20_000)
# do(ctrap(d=0.05/0.2, loc=2.9, scale=0.2), N=500) # Note won't work first time through

In [ ]:
# Exercise: Compute summary statistics from the samples

# ONE_SIGMA is the confidence width 68% interval from -1σ to 1σ
ONE_SIGMA = scipy.stats.norm.cdf(1) - scipy.stats.norm.cdf(-1)
print(f"1-σ interval: {ONE_SIGMA*100:.2f}%")


def my_print_stats(samples, logp=None):
    def fi(interval):  # interval formatter
        return f"[{interval[0]:.3f}, {interval[1]:.3f}]"

    # mean = ...
    # std = ...
    # median = ...
    # p95 = ...
    # p68 = ...
    # hdi = ... # highest density interval
    # entropy = ...
    # print(f"mean: {U(mean, std):.3uS} {median=:.4f} p95={fi(p95)} p68={fi(p68)} hdi={fi(hdi)} {entropy=:.4f}")
    # px.histogram(samples, marginal="rug", histnorm="probability density").show()


dist = scipy.stats.norm(0, 1)
# dist=scipy.stats.chi2(3)
# dist=scipy.stats.poisson(5)

samples = dist.rvs(size=5000)
my_print_stats(samples, dist.logpdf(samples))

### Skewness and reporting values

Not all distributions are symmetric normal distributions.

The next cells let you explore some different distributions, showing you the statistics from the sample along with the predicted values.

The mean, median and mode differ when the distribution is skewed.

Play with the number of samples. Uncertainty in the MC integrals should scale as $√N$, with an additional factor due to the variance in the sample values.

When the distribution is not normal, the NIST guidance is to report the center and width of the credible interval. Regardless of the choice of interval (usually 1-σ, 95% or 99%), convert it to the 1-σ equivalent and scale it to $1u_c$ or $2u_c$ according to the needs of the publication. See §4.3 of [NIST TN 1297](https://www.nist.gov/pml/nist-technical-note-1297) *Guidelines for Evaluating and Expressing the Uncertainty of NIST Measurement Results*.

[Note: My implementations of HDI (highest density interval) give the wrong answer for the poisson distribution. Fixes welcome!]


In [ ]:
# Code for CTrap distribution  *** SKIP ***
# Based on equations in http://dx.doi.org/10.1088/0026-1394/46/3/012
# This is used in some of the GUM S1 examples

from scipy.stats._distn_infrastructure import (
    _vectorize_rvs_over_shapes,
    get_distribution_names,
    _kurtosis,
    _isintegral,
    rv_continuous,
    _skew,
    _get_fixed_fit_value,
    _check_shape,
    _ShapeInfo,
)


def _ctrap_pdf(x, d):
    a, b = 0, 1
    mid, w = (b + a) / 2, (b - a) / 2
    g = np.log((w + d) / np.maximum(abs(x - mid), w - d))
    return np.clip(g, 0, np.inf) / (4 * d)


def _ctrap_cdf(x, d):
    μ, δ, ε = 0.5, 0.5, d
    ξ = x - μ
    δp, δm = δ + ε, δ - ε
    F = np.empty(shape=x.shape, dtype="d")
    # left side
    F[ξ <= -δp] = 0.0
    # curvilinear rise
    mask = (-δp < ξ) & (ξ < -δm)
    # print(mask.shape, F.shape, δp, ε.shape)
    t1 = ξ * (np.log(δp / abs(ξ)) + 1)
    F[mask] = ((t1 + δp) / (4 * ε))[mask]
    # flat top
    mask = (-δm <= ξ) & (ξ <= δm)
    F[mask] = ((ξ * np.log(δp / δm) + 2 * ε) / (4 * ε))[mask]
    # curvilinear fall
    mask = (δm < ξ) & (ξ < δp)
    F[mask] = ((t1 + 3 * ε - δ) / (4 * ε))[mask]
    # right
    F[ξ >= δp] = 1.0
    return F


def _ctrap_rvs(d, size=None, random_state=None):
    r1 = random_state.uniform(size=size)
    r2 = random_state.uniform(size=size)
    a, b = 0, 1
    a_s = (a - d) + 2 * d * r1
    b_s = (a + b) - a_s
    x = a_s + (b_s - a_s) * r2
    return x


class ctrap_gen(rv_continuous):
    """curvilinear trapezoid distribution"""

    def _shape_info(self):
        ia = _ShapeInfo(name="d", integrality=False, domain=(0, 0.5), inclusive=(True, True))
        return [ia]

    def _rvs(self, d, size=None, random_state=None):
        return _ctrap_rvs(d, size=size, random_state=random_state)

    def _pdf(self, x, d):
        return _ctrap_pdf(x, d)

    def _cdf(self, x, d):
        return _ctrap_cdf(x, d)

    def _stats(self, d):
        return (
            0.5,  # mean
            1 / 12 + d**2 / 9,  # variance
            0,  # no skewness
            np.nan,  # wrong excess kurtosis
        )

    def median(self, d, loc=0, scale=1):
        return 0.5 * scale + loc


ctrap = ctrap_gen(name="ctrap", longname="curvilinear trapezoid")

# dist = ctrap(d=0.05/0.2, loc=2.9, scale=0.2) # x = 3.0±0.1 with uncertainty rounded
# dist_stats(dist)

In [ ]:
# Helper functions for exploring distributions *** SKIP ***

# print_stats(samples, logp) implements the monte carlo integrals listed above
# dist_stats(dist) shows the predicted values of the stats

# Exercise: decorate the histograms with critical values such as mean, median, mode, and credible intervals


def dist_mode(dist):
    """
    Return the modal value of the distribution, or a modal value if it is multi-modal.
    """
    # Find the peak probability given percentile so we can do a bounded search over [0, 1]
    if hasattr(dist, "pdf"):

        def f(x):
            return -dist.pdf(dist.ppf(x))
    else:

        def f(x):
            return -dist.pmf(dist.ppf(x))

    result = scipy.optimize.minimize_scalar(
        fun=f,
        bounds=(0, 1),
        method="bounded",
        options={"maxiter": 150},
    )
    return dist.ppf(result.x)


def dist_hdi(dist, ci):
    """
    Find the highest density interval in a distribution.

    *dist* is a frozen distribution from scipy.stats
    *ci* is the confidence interval covered by the HDI.

    TODO: giving the wrong value for Poisson(5)
    """

    def f(x):
        w = dist.ppf(x + ci) - dist.ppf(x)
        # Handle discrete distributions.
        # Don't use inf because inf-inf is undefined.
        return w if w > 0 else 1e100

    result = scipy.optimize.minimize_scalar(
        fun=f,
        # x0=(1-width)/2,
        bounds=(0, 1 - ci),
        method="bounded",
        options={"maxiter": 150},
    )
    return (dist.ppf(result.x), dist.ppf(result.x + ci))


def dist_stats(dist):
    """print stats for a scipy.stats distribution"""
    print(f"== statistics for {dist.dist.name}{dist.args}")
    mean, std = dist.mean(), dist.std()
    median = dist.median()
    mode = dist_mode(dist)
    p95 = dist.interval(0.95)
    p68 = dist.interval(ONE_SIGMA)
    hdi = dist_hdi(dist, ONE_SIGMA)
    entropy = dist.entropy()

    def fi(interval):
        return f"[{interval[0]:.4f}, {interval[1]:.4f}]"

    print(
        f"mean: {U(mean, std):.3uS} {median=:.4f} {mode=:.4f} p95={fi(p95)} p68={fi(p68)} hdi={fi(hdi)} H={entropy:.4f}"
    )


# ONE_SIGMA is the confidence width 68% interval from -1σ to 1σ
ONE_SIGMA = scipy.stats.norm.cdf(1) - scipy.stats.norm.cdf(-1)
print(f"1-σ interval: {ONE_SIGMA*100:.2f}%")


def nbins_fd(data):
    """Estimate number of histogram bins based on IQR (Freedman-Diaconis rule)."""
    n = len(data)
    # q75, q25 = np.percentile(data, [75 ,25])
    # assume data is sorted for this notebook
    q75, q25 = data[(3 * n) // 4], data[n // 4]
    iqr = q75 - q25
    bin_width = 2 * iqr * n ** (-1 / 3)
    if bin_width == 0:
        return 1
    nbins = 1 if bin_width == 0 else int(np.ceil((data.max() - data.min()) / bin_width))
    # print(f"{iqr=:.4f} {n=} {bin_width=:.4f} range={data.max() - data.min():.4f} {nbins=}")
    return nbins


def print_stats(samples, logp=None, ci=0.99, hdi_p=ONE_SIGMA):
    """
    Generate statistics from the samples. Print them and draw the histogram. If *logp* is provided
    the compute entropy and plot the PDF on top of the histogram.

    The *logp* plot only works if this is sampling from a one dimensional distribution. More generally
    we would need to integrate the probabilities within each bin and scale by the total integrated
    probability. There would also need to be a bin-size correction to turn it into a probability density.
    See the difference between histnorm of 'density', 'probability' and 'probability density' in plotly.
    """
    # Need to know if it is a continuous distribution for plotting histogram
    # Do this before sorting so that we don't risk extreme values accidentally
    # presenting as integers.
    continuous = (samples[:10] != np.floor(samples[:10])).any()

    N = len(samples)

    # Simple MC integration for mean
    mean = np.sum(samples) / N

    # std uses the bias correction
    std = np.sqrt(np.sum((samples - mean) ** 2) / (N - 1))  # unbiased

    # Sort the samples to get the quantiles
    index = np.argsort(samples)
    samples = samples[index]
    if logp is not None:
        logp = logp[index]
    median = samples[N // 2]  # could average central pair if N is odd

    # Estimate 95% and 1-σ equal tail intervals (ETI) using a fixed window
    p95 = (samples[w := int(N * 0.025)], samples[-w])
    p68 = (samples[w := int(N * (1 - ONE_SIGMA) / 2)], samples[-w])

    # Estimate 1-σ highest density interval (HDI) using a sliding window
    width = int(np.ceil(N * hdi_p))
    delta = samples[width:] - samples[:-width]
    a = np.argmin(delta)
    hdi = (samples[a], samples[a + width])

    # Simple MC integration for entropy if we have logp, otherwise use GMM
    if logp is not None:
        entropy = -np.sum(logp) / N
    else:
        # Unknown likelihood. Instead compute entropy from a gaussian mixture model
        from bumps.dream.entropy import gmm_entropy

        H, dH = gmm_entropy(samples[:, None])
        entropy = U(H, dH)

    # Get mode directly from logp
    if logp is not None:
        mode_index = np.argmax(logp)
        mode = samples[mode_index]
    else:
        mode = np.nan  # we could fit the mode from the gmm

    # Print stats
    def fi(interval):  # interval formatter
        return f"[{interval[0]:.4f}, {interval[1]:.4f}]"

    print(f"== sampled {N=}")
    print(
        f"mean: {U(mean, std):.3uS} {median=:.4f} {mode=:.4f} p95={fi(p95)} p68={fi(p68)} hdi{int(100*hdi_p)}={fi(hdi)} H={entropy:.4f}"
    )

    # Generate histogram using 99% ETI
    outliers = int((1 - ci) / 2 * N)
    trimmed = samples[outliers:-outliers] if outliers > 0 else samples
    # print(f"{N=} {outliers=} {samples.shape=} {trimmed.shape=}")
    # Hist bins for discrete distributions should use integer spacing
    nbins = nbins_fd(trimmed) if continuous else int(trimmed[-1] - trimmed[0] + 1)
    # nbins = None # Let px.histogram decide the number of bins
    # trimmed = samples  # no trimming
    fig = px.histogram(
        trimmed,
        nbins=nbins,
        opacity=0.7,
        histnorm="probability density",
        # marginal='rug',
        title=f"Histogram of the {100*ci:g}% equal tail interval",
        labels={"value": "x", "count": "Probability density"},
        # name="samples",
    )
    fig.update_traces(name="measured", selector=dict(type="histogram"))

    # If we have logp then show the probability curve on top of the histogram.
    if logp is not None:
        # Show the probability curve using 400 points from the pdf/pmf
        # This works because we sorted by samples value
        pdf = np.exp((logp[outliers:-outliers] if outliers > 0 else logp))
        # steps = slice(None, None, (len(pdf)+399)//1000)
        steps = slice(None)
        # Note: "correcting" pdf to match the probability density for the trimmed data
        fig.add_trace(go.Scatter(x=trimmed[steps], y=pdf[steps] / ci, name="theory"))
    # fig.update_layout(legend_title_text=None)
    fig.update_layout(showlegend=False)
    fig.show()

In [ ]:
# Exercise: vary distributions and shape parameters to see the effect on credible intervals

ci = 0.99
# dist=scipy.stats.norm(loc=10,scale=0.2)
dist = scipy.stats.chi2(3)
ci = 0.995  # skewed
# dist=scipy.stats.poisson(5); ci=1  # Note: wrong values for hdi
dist = scipy.stats.poisson(5)
ci = 1
# dist=scipy.stats.poisson(0.2); ci=1
# dist=scipy.stats.beta(0.6, 0.5); ci=0.95 # multi-modal
# dist=scipy.stats.beta(5, 1); ci=1
# dist=scipy.stats.beta(1, 3); ci=1
# dist=scipy.stats.beta(2, 5); ci=1
# dist=scipy.stats.cauchy(loc=40, scale=0.35); ci=0.90 # mean and variance undefined


## Distributions defined in GUM S1 §6.4. See the cell below for more details.
## trapezoid and triang have X = X1 + X2, with X1 ~ U(a1, a1+w1) and X2 ~ U(a2, a2+w2)
## For trapezoid, widths are unequal. Use a1=3, a2=7, w1+w2=10, |w1-w2|/(w1+w2)=0.3
## For triangular widths are equal. Use a1=3, a2=7, w1=w2=5

# dist = scipy.stats.uniform(loc=2.5, scale=3.5-2.5); ci=1 # x is reported as 3, so use U[2.5, 3.5] for the range
# dist = ctrap(d=0.05/0.2, loc=2.9, scale=0.2); ci=1 # x = 3.0±0.1 with uncertainty rounded
# u=0.1; dist = ctrap(d=0.05/(2*u), loc=3-u, scale=2*u); ci=1 # x = 3.0±u with uncertainty rounded
# u=0.1; dist = ctrap(d=0.005/(2*u), loc=3-u, scale=2*u); ci=1 # x = 3.0±u with uncertainty rounded to 2 digits
# dist = scipy.stats.trapezoid((1-0.3)/2, (1+0.3)/2, loc=3+7, scale=10); ci=1 # x is /x1+x2 uniform of unequal width
# dist = scipy.stats.triang(0.5, 3+7, scale=2*5); ci=1  # x is x1+x2 uniform of the same width
# dist = scipy.stats.arcsine(loc=26, scale=28-26); ci=0.95 # x is at 27±1 °C with sinusoidal control
# dist = scipy.stats.norm(loc=24, scale=0.5) # x is reported as 24 ± 0.5 with 1-σ uncertainty
# dist = scipy.stats.t(10-1, loc=24, scale=0.5/np.sqrt(10)) # x is the average of n=10 observations with μ=24, σ=0.5
# dist = scipy.stats.expon(scale=37) # x is known to be non-negative with a reported value of 37
# dist = scipy.stats.gamma(12+1, scale=1); ci=1 # x is the distribution of rates yielding Poisson count q=12
# dist = scipy.stats.norm(12, scale=np.sqrt(12)); ci=1 # pretend Poisson is normal with q ± √q

# Sample from the distribution and show the stats from the sample
# Compare different sizes of N 1_000, 10_000, 30_000, 100_000, 1_000_000
samples = dist.rvs(1_000_000)
logp = dist.logpdf(samples) if hasattr(dist, "logpdf") else dist.logpmf(samples)
dist_stats(dist)
print_stats(samples, logp=logp, ci=ci)

### Forward uncertainty propagation

GUM Supplement 1 provides guidelines for propagating uncertainty in a set of measurements through an expression to generate a derived value.

You first need to assign a distribution to each input value. You then draw the same number of samples from each input and run them through the derived expression. The resulting $Y$ values is the posterior distribution which you can use to estimate $\hat y$ and its uncertainty.

A list of distributions for common circumstances is given in §6.4, and reproduced here.

In [ ]:
# Figure generation for common priors  *** SKIP ***

import numpy as np, scipy.stats as st, matplotlib.pyplot as plt, io, base64, sys

# list of (label, distribution)
# uniform on [2.5, 3.5] but our x is 0-1, pdf will be zero; maybe they want scaled? We'll just include as example.


counts = 12

dists = [
    (scipy.stats.uniform(loc=2.5, scale=3.5 - 2.5), "x = 3, uncertainty unknown"),
    (ctrap(d=0.05 / (2 * 0.1), loc=3 - 0.1, scale=2 * 0.1), "x = 3.0(1) uniform, 3.00(10) in orange"),
    (scipy.stats.trapezoid((1 - 0.3) / 2, (1 + 0.3) / 2, loc=3 + 7, scale=10), "x = u1 + u2 uniform of unequal width"),
    (scipy.stats.triang(0.5, 3 + 7, scale=2 * 5), "x = u1 + u2 uniform of equal width"),
    (scipy.stats.arcsine(loc=26, scale=28 - 26), "x = 27±1 °C with sinusoidal control"),
    (scipy.stats.norm(loc=3, scale=0.1), "x = 3.0(1) normal, uniform in orange"),
    (scipy.stats.t(10 - 1, loc=24, scale=0.5 / np.sqrt(10)), "μ = 24±0.5 using 10 IID normal values"),
    (scipy.stats.expon(scale=37), "x = 37 with x non-negative"),
    (scipy.stats.gamma(counts + 1, scale=1), f"x = {counts} Poisson counts, gaussian in orange"),
    # (scipy.stats.norm(41, scale=np.sqrt(41)), "x = 41±√41 Poisson normal approximation"),
]

# 3️⃣  Prepare a 3×3 grid of sub‑plots
n = len(dists)
cols = 3
rows = (n + cols - 1) // cols  # ceiling division
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3), constrained_layout=True)

# flatten the axes array for easy iteration
axes = axes.ravel()

for ax, (dist, gloss) in zip(axes, dists):
    name = dist.dist.name
    if name == "arcsine":
        x = dist.ppf([0.05, 0.95])
        left, right = x
    elif name == "gamma":
        left, right = dist.ppf([0.00001, 0.999])
    else:
        x = dist.ppf([0, 0.005, 0.995, 1])  # inverse CDF
        left = x[0] if np.isfinite(x[0]) else x[1]
        right = x[-1] if np.isfinite(x[-1]) else x[-2]
        w = 0.05 * (right - left)
        left, right = left - w, right + w
        # print(dist.dist.name, left, right, w)
    x = np.linspace(left, right, 400)
    y = dist.pdf(x)  # PDF at those points

    args = [f"{v:g}" for v in dist.args] + [f"{k}={v:g}" for k, v in dist.kwds.items()]
    label = f"{gloss}\n{dist.dist.name}({', '.join(args)})"
    ax.plot(x, y)
    if name == "gamma":
        ax.plot(x, scipy.stats.norm(counts, np.sqrt(counts)).pdf(x))
    elif name == "ctrap":
        ax.plot(x, ctrap(d=0.005 / (2 * 0.1), loc=3 - 0.1, scale=2 * 0.1).pdf(x))
    elif name == "norm":
        ax.plot(x, ctrap(d=0.05 / (2 * 0.1), loc=3 - 0.1, scale=2 * 0.1).pdf(x))
    # ax.set_xlabel("x")
    # ax.set_ylabel("pdf")
    ax.set_title(label)
    # ax.legend(fontsize="x-small")
    ax.grid(alpha=0.3)

fig.suptitle("Common priors for forward error propagation", fontsize=20)

None

Common distributions for forward uncertainty propagation:

* rectangular: $X \sim \text{U}[a, b]$ uses `stats.uniform(a, b)`.<br> For example a value $x$ is reported to be 850 with no additional information. Assuming that this is a rounded result from a more precise measurement, use $X \sim \text{U}[845, 855]$ in the derived expression.

* symmetric trapezoid: $X \sim \text{Trap}[a, b, β]$ uses `stats.trapezoid(β/2, 1-β/2, loc=a, scale=(b-a))`.<br>Generated from a sum of two uniform values, $X_1 \sim U(a_1, b_1)$ and $X_2 \sim U(a_2, b_2)$, giving base $[a=a_1+a_2, b=b_1+b_2]$ and top to base ratio $β=|(b_1-a_1) - (b_2-a_2)| / (b-a)$.

* symmetric triangular: $X \sim \text{Tri}[a, b]$ uses `stats.triang(0.5, loc=a, scale=b-a)$.<br>Generated from a sum of two uniform values, $X_1 \sim U(a_1, b_1)$ and $X_2 \sim U(a_2, b_2)$ with the same width $b_k-a_k$, giving base $[a=a_1+a_2, b=b_1+b_2]$ and peak at $(a+b)/2$.

* curvilinear trapezoid: $X \sim \text{CTrap}[a, b, d]$ uses two uniform values and the expression in §6.4.3.4 to from a value for $x$.<br>The example given is a voltage reading of 10.0 V ± 0.1 V. Since the uncertainty is rounded to 0.1 V, the actual uncertainty is $d ~ \text{U}[0.05, 0.15]$ and so the value is somewhere in $[10.0-0.15, 10.0+0.15]$ with a curvilinear slope from 10.0-0.15 to 10.0-0.05 and again from 10.0+0.05 to 10.0+0.15, with a flat top between. Using two digits for uncertainties below 0.3 would have avoided this problem. Above 0.3, the additional uncertainty from rounding can be ignored.

* arcsine: $X \sim \text{Arcsine}(a, b) = (b+a)/2 + (b-a)/2 \sin(Φ)$ for $Φ \sim \text{U}[0, 2π]$ uses `stats.arcsine(loc=a, scale=b-a)`.<br>If $Δ$ is known to be sinusoidal over $[a, b]$ but the phase is unknown, then $Δ$ is distributed as arcsine. For example (§9.5.2.8) temperature $T$ = 19.9 ± 0.5 °C, so use $T = 19.9 + Δ$ in the derived expression, with $Δ \sim \text{Arcsine}(-0.5, 0.5)$

* normal: $X \sim \text{N}(x, u_c^2)$ uses `stats.norm(loc=x, scale=u_c)`.<br>If $x$ has uncertainty $u_c$, then $x$ is normally distributed.

* MVN: $X \sim \text{N}(\bm{μ}, \bm{Σ})$ uses `stats.multivariate_normal(mean=μ, cov=Σ)`.<br>If $x$ are correlated parameters with covariance $Σ$, then $x$ is distributed as a multivariate normal.

* t: $(X - \hat μ) / (\hat σ/√n) \sim \text{t}(n-1)$ uses `stats.t(n-1, loc=μ, scale=σ/√n)`.<br>Given n observations $x_i \sim N(μ_0, σ_0^2)$ IID with unknown $μ_0$ and $σ_0^2$, then the measured $μ$ follows a $t$ distribution.

* exponential: $X \sim \text{Exp}(1/x)$ uses `stats.expon(loc=0, scale=x)`.<br>If it is only known that the measurement $x$ comes from non-negative $X$, then $x$ follows an exponential distribution.

* gamma: $X \sim \text{Γ}(q+1, 1)$ uses `stats.gamma(q+1, loc=0, scale=1)`.<br>If the observed counts from a Poisson process is $q$, then the Poisson rate parameter follows a gamma distribution. This is skewed a bit high with a heavier tail than
the normal approximation. Especially for low count rate experiments, use a gamma distribution to properly analyze your data.


### Mass calibration example


Forward uncertainty propagation for the GUM‑S1 §9.3 mass‑calibration example.

Consider the calibration of a weight $W$ of mass density $ρ_W$ against a reference weight $R$ of mass density $ρ_R$ having nominally the same mass, using a balance operating in air of mass density $ρ_a$

The equation is as follows:

> $δm = (m_{R,c} + δm_{R,c}) * [1 + (ρ_a - ρ_{a0}) * (1/ρ_W - 1/ρ_R)] - m_\text{nom}\quad\quad\quad(24)$

Table 5 – Input quantities $X_i$ and the PDFs assigned to them for the mass‑calibration model (24) (9.3.1.4)

| Parameter      | Distribution | Expectation $E[X_i]$                | Uncertainty | Description |
|----------------|--------------|-------------------------------------|-------------|-------------|
| $m_{R,c}$      | $N(μ,σ^2)$   | 100 000.000 mg                      | 0.050 mg    | conventional reference mass |
| $δm_{R,c}$     | $N(μ,σ^2)$   | 1.234 mg                            | 0.020 mg    | balancing reference |
| $ρ_{a}$        | $R(a,b)$     | 1.20 $\text{kg m}^{-3}$             | 0.10 $\text{kg m}^{-3}$ | air density |
| $ρ_{a0}$       | exact        | 1.20 $\text{kg m}^{-3}$             | | conventional air density |
| $ρ_{W}$        | $R(a,b)$     | 8 × $10^3$ $\text{kg m}^{-3}$       | 1 × $10^3$ $\text{kg m}^{-3}$ | calibration density |
| $ρ_{R}$        | $R(a,b)$     | 8.00 × $10^3$ $\text{kg m}^{-3}$    | 0.05 × $10^3$ $\text{kg m}^{-3}$ | reference density |
| $m_\text{nom}$ | exact | 100 g | | nominal mass |


Generate M = 1 000 000 draws from the prior distributions, evaluate the model, then show statistics on the posterior density of $δm$.

Reported values: $δm$ = 1.2341(754), 95% HDI [1.0834, 1.3825]

In [ ]:
# Exercise: perform the monte carlo method on the above example

...

if 0:
    # Show correlations
    samples = np.column_stack((...))
    labels = [...]
    corner.corner(samples, labels=labels)

In [ ]:
# Mass calibration example solution (coding assistance from AI)
"""
Forward error‑propagation for the GUM‑Supplement 1 §9.3 mass‑calibration example
(eq. 24).

We generate M = 1 000 000 draws from the prior (uniform) distributions,
evaluate the model

    δm = (m_R + δm_R) * [1 + (ρ_a – ρ_a0) * (1/ρ_W – 1/ρ_R)] – m_nom

and store the results in a ``bumps.dream.state.State`` object so they can be
handled downstream by any Bumps/Dream tools (e.g. plotting, uncertainty
analysis, etc.).
"""

# ----------------------------------------------------------------------
# 1️⃣  Imports
# ----------------------------------------------------------------------
import numpy as np
from typing import Any

# Bumps/Dream objects – the State container holds the MCMC draws.
# (The State class is lightweight; we only need the ``samples`` attribute here.)
from uncertainties import ufloat as U


def stats(x):
    return U(np.mean(x), np.std(x))


# ----------------------------------------------------------------------
# 2️⃣  Fixed constants (nominal values & their uncertainties)
# ----------------------------------------------------------------------
m_R = 100_000.000  # mg (reference mass)
u_m_R = 0.050  # mg (1‑sigma uncertainty)

δm_R = 1.234  # mg (mass offset)
u_δm_R = 0.020  # mg (1‑sigma uncertainty)

m_nom = 100_000.0  # mg the “nominal” mass we subtract

ρ_a0 = 1.2  # kg m⁻³ (air density at reference condition – exact)

# ----------------------------------------------------------------------
# 3️⃣  Prior distributions (uniform as given)
# ----------------------------------------------------------------------
M = 1_000_000  # number of Monte‑Carlo draws

# Using numpy RNG rather than scipy.stats
rng = np.random.default_rng()

# air density ρ_a  ~ U(1.10, 1.30)   kg m⁻³
ρ_a = rng.uniform(1.10, 1.30, size=M)

# water density ρ_W ~ U(7000, 9000) kg m⁻³
ρ_W = rng.uniform(7_000, 9_000, size=M)

# reference density ρ_R ~ U(7950, 8050) kg m⁻³
ρ_R = rng.uniform(7_950, 8_050, size=M)

# ----------------------------------------------------------------------
# 4️⃣  Propagate uncertainties for the *fixed* quantities
# ----------------------------------------------------------------------
# We treat m_R and δm_R as normal (Gaussian) random variables because their
# uncertainties are quoted as standard deviations.
m_R_samples = rng.normal(loc=m_R, scale=u_m_R, size=M)  # mg
δm_R_samples = rng.normal(loc=δm_R, scale=u_δm_R, size=M)  # mg

# ----------------------------------------------------------------------
# 5️⃣  Evaluate the forward model (eq. 24) for every draw
# ----------------------------------------------------------------------
# The term in brackets is dimensionless.
bracket = 1.0 + (ρ_a - ρ_a0) * (1.0 / ρ_W - 1.0 / ρ_R)

δm = (m_R_samples + δm_R_samples) * bracket - m_nom  # mg
print(f"δm = {stats(δm):.3uS} mg [target value is {U(1.2341,0.0754):.3uS}]")

# ``samples`` is a 2‑D array: (n_draws, n_parameters).  Here we keep only the
# model output (δm) as the first column and optionally the raw inputs as
# additional columns for diagnostic purposes.
samples = np.column_stack(
    [
        δm,  # model output (Δm)
        m_R_samples,  # reference mass draws
        δm_R_samples,  # mass offset draws
        ρ_a,
        ρ_W,
        ρ_R,  # the three density draws
    ]
)


logp = (
    -0.5 * ((m_R_samples - m_R) / u_m_R) ** 2
    - 0.5 * ((δm_R_samples - δm_R) / u_δm_R) ** 2
    - np.log(0.2)
    - np.log(2_000)
    - np.log(100)
)

# (Optional) give each column a name – this helps Bumps plotting utilities.
labels = [
    "Δm (mg)",  # model output
    "m_R (mg)",
    "δm_R (mg)",
    "ρ_a (kg/m³)",
    "ρ_W (kg/m³)",
    "ρ_R (kg/m³)",
]

print_stats(δm, logp=None, hdi_p=0.95)

corner.corner(samples, labels=labels)

None

### How many samples is enough?

Testing on a draw from a gaussian distribution, the 1-σ coverage intervals should be [-1, 1].

Switching to a skew distribution, chisq(3) shortest=3.4901 and symmetric=4.3515

For the 95% interval, chisq(3) shortest=7.8140 and symmetric=9.1326

The results for various $M$ as recorded on a [github issue](https://github.com/bumps/bumps/issues/452#issuecomment-4911345584) are shown here:

### Gaussian: 1-σ shortest=2 symmetric=2
| $M$ | Shortest | Symmetric | Robust | Robust Shortest |
| ---: | :--- | :--- | :--- | :--- |
| 1,000,000 | 1.9995(19) | 1.9998(19) | 2.0042(20) | 2.0040(19) |
| 100,000 | 1.999(6) | 2.000(7) | 2.015(7) | 2.013(6) |
| 30,000 | 1.997(11) | 2.000(12) | 2.026(12) | 2.023(11) |
| 10,000 | 1.994(19) | 2.000(20) | 2.045(20) | 2.039(19) |
| 1,000 | 1.98(6) | 2.00(6) | 2.15(6) | 2.13(6) |
| 100 | 1.96(18) | 1.99(19) | 2.50(22) | 2.50(21) |

### χ²-3: 1-σ shortest=3.4901 and symmetric=4.3515
| $M$ | Shortest | Symmetric | Robust | Robust Shortest |
| ---: | :--- | :--- | :--- | :--- |
| 1,000,000 | 3.4913(27) | 4.352(4) | 4.362(4) | 3.4999(28) |
| 100,000 | 3.489(12) | 4.351(17) | 4.384(17) | 3.516(12) |
| 30,000 | 3.488(21) | 4.352(31) | 4.412(31) | 3.538(22) |
| 10,000 | 3.48(4) | 4.35(5) | 4.46(5) | 3.57(4) |
| 1,000 | 3.47(12) | 4.35(17) | 4.69(18) | 3.77(12) |


### χ²-3: 95% shortest=7.8140 and symmetric=9.1326
| $M$ | Shortest | Symmetric | Robust | Robust Shortest |
| ---: | :--- | :--- | :--- | :--- |
| 1,000,000 | 7.813(9) | 9.132(16) | 9.156(17) | 7.836(9) |
| 100,000 | 7.814(30) | 9.13(4) | 9.21(4) | 7.886(30) |
| 30,000 | 7.81(6) | 9.13(8) | 9.27(8) | 7.95(6) |
| 10,000 | 7.81(10) | 9.13(14) | 9.38(14) | 8.05(10) |
| 1,000 | 7.84(31) | 9.1(4) | 10.0(5) | 8.7(4) |


Relative uncertainty the stats for M=30,000 at the 95% interval are within 1%. Pushing it to M=1,000,000 drops the uncertainty to 0.1%. These are the δ tolerance for n=1 and 2 digits given above. Similarly for the 1-σ interval.


GUM-S1 discusses 

Note that uncertainty in the value of the mean $s_μ$ for is different from the width of the distribution $σ = u(y)$. As you increase the number of samples, $s_μ$ decreases, as does the uncertainty in the width $s_σ$ but the width itself stays the same.

The key value for setting the precision is the width of the distribution. For narrow distributions you will want many digits of precision for the mean and the interval, but for a broad distribution it doesn't matter. In practice, with narrower distributions the points are clustered and high precision is easier to attain.

The precision on the 1-σ interval should be higher than the 95% interval. This is because precision decreases as the variance increases. Since the points are sampled less densely at the boundaries of the 95% interval, the variance in the values will increase and drive up the Monte Carlo uncertainty. 


In [ ]:
# Exercise: verify claim that required samples is independent of distribution

# Modify the stats function to return stats of interest
#    HDI-95 endpoints, HDI-68 endpoints, mean, std, median
# Run a loop collecting stats for different runs of the same distribution
# Generate uncertainties on each of these stats
# Repeat for different combinations of location, scale and skewness

# Expectation: 10,000 samples gives you 1% Δω/ω, 1,000,000 gives you 0.1%


## Multi-dimensional distributions

To view a multidimensional distribution we used a corner plot

The individual plots are low dimensional projections of the joint probability distribution.

The diagonal elements show the histogram for a single dimension 

The off-diagonal elements $(i,j)$ show the correlation between $i$ and $j$

It is an ugly looking integral: $P(ω_k) = \int_{Ω_{j \ne k}} P(ω_k | ω_j, j \ne k) P(ω_j, j \ne k)\, \mathrm{d}{Ω_{j \ne k}}$

... but it is trivial to implement: just look at single columns

The sampling density for the single column captures the integrated weights of all the other columns

Similarly for a pair of columns $(i, j)$ and the off-diagonal correlation plot

Try it for the Dirichlet distribution [scipy.stats.dirichlet](https://docs.scipy.org/doc/scipy-1.11.4/reference/generated/scipy.stats.dirichlet.html)

In [ ]:
# Exercise: plot a sample from the dirichlet distribution


In [ ]:
alpha = np.array([0.4, 5, 15, 6, 3])  # specify concentration parameters
alpha = np.random.rand(5)
print("α =", alpha)
dist = scipy.stats.dirichlet(alpha)
samples = dist.rvs(size=10000)
corner.corner(samples)

None  # Need None here otherwise the plot is shown twice